# Desktop Voice Orchestrator — M3 个性化 ASR 扩词与个人音色克隆防失真训练流水线 (`asr-personal-v2`)

本 Notebook 在 **Google Colab (推荐 L4 或 A100 GPU)** 上一键完成 M3 阶段全流程：
1. **Cell 1**：挂载 Google Drive，检查 GPU 与云盘数据完整性（`dataset/manifest.jsonl`、黄金参考音频池 `prompt_pool/`、扩词合成清单）；
2. **Cell 2**：安装环境依赖（`CosyVoice2`、`sherpa-onnx`、`funasr`、`kaldi-native-fbank`、`onnxruntime-gpu`、`faster-whisper`、`pypinyin`）并拉取基础模型；
3. **Cell 3（先试听把关）**：运行 `--mode pilot` 生成 **25 条** 个人音色克隆试听小样，直接在页面内渲染 `IPython.display.Audio` 播放器与独立裁判通过率报告；
4. **Cell 4（可选：处理今晚游戏录音）**：若云盘上传了戴耳机游戏会话，调用 `mine_game_sessions.py --transcribe` 执行 Silero VAD 切分 + `SenseVoice` / `Whisper large-v3` 双模型 100% 逐字一致交叉打标与正负样本分流；
5. **Cell 5（批量扩词造数）**：试听满意后，运行全量 **3,600 条** 零样本克隆合成 + 独立裁判闭环过滤（支持 `--resume` 断点续跑）；
6. **Cell 6（微调与导出 `asr-personal-v2`）**：以本人真机录音为主锚点（`--repeat 4`）+ 每轮动态抽样合成语料（`--tts-max-per-epoch 1800`）+ `0~15 dB` 真实音乐混音，解冻编码器后 3 层微调并导出 `QUInt8` `model.int8.onnx` 到云盘 `models/asr-personal-v2/`，输出完整离线评测对比表。

## Cell 1：挂载 Google Drive 并检查 GPU 与目录完整性

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/desktop-voice-orchestrator-data")
REPO_DIR = Path("/content/desktop-voice-orchestrator")
REPO_URL = "https://github.com/psyduck-10385/desktop-voice-orchestrator.git"
REPO_BRANCH = "feat/m1-m3-foundation"

print("=== 1. GPU 硬件检查 ===")
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True)
print(gpu_info.stdout.strip() if gpu_info.returncode == 0 else "⚠️ 未检测到 NVIDIA GPU，请切换运行时到 L4 或 A100！")

print("\n=== 2. 定位训练脚本仓库 ===")
if (DRIVE_ROOT / "repo_sync" / "training" / "asr" / "finetune.py").is_file():
    REPO_DIR = DRIVE_ROOT / "repo_sync"
    print(f"✅ 使用云盘同步仓库目录: {REPO_DIR}")
elif not REPO_DIR.is_dir():
    print(f"正在克隆仓库 {REPO_URL} ({REPO_BRANCH}) ...")
    rc = subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)]).returncode
    if rc != 0:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=False)
print(f"当前工作区路径: {REPO_DIR}")

print("\n=== 3. 云盘数据目录完整性体检 ===")
checks = {
    "云盘数据根目录": DRIVE_ROOT,
    "录音切片目录 (dataset/clips)": DRIVE_ROOT / "dataset" / "clips",
    "录音主清单 (dataset/manifest.jsonl)": DRIVE_ROOT / "dataset" / "manifest.jsonl",
    "本人黄金参考音频池 (prompt_pool.jsonl)": DRIVE_ROOT / "m3_tts_prep" / "prompt_pool" / "prompt_pool.jsonl",
    "25条试听合成清单 (pilot_prompts_25.jsonl)": DRIVE_ROOT / "m3_tts_prep" / "pilot_prompts_25.jsonl",
    "3600条全量扩词清单 (full_prompts_3600.jsonl)": DRIVE_ROOT / "m3_tts_prep" / "full_prompts_3600.jsonl",
}
for label, p in checks.items():
    exists = p.exists()
    mark = "✅ 就绪" if exists else "⚠️ 缺失 (请确认本机已生成并完成 Google Drive 同步)"
    print(f"  {mark} | {label}: {p}")

if not (DRIVE_ROOT / "dataset" / "manifest.jsonl").is_file():
    print("\n💡 提示：Cell 3~5（个人音色克隆试听与全量造数）现在即可直接运行！")
    print("   在进入 Cell 6（微调训练）前，请确保本机导出的 `dataset/manifest.jsonl` 已同步至云盘 `dataset/` 目录。")

## Cell 2：一键安装环境依赖并拉取三大基础模型

In [ ]:
import os
import subprocess
import tarfile
import urllib.request
from pathlib import Path

MODELS_ROOT = Path("/content/models")
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

print("=== 1. 安装核心 Python 依赖 ===")
pip_pkgs = [
    "sherpa-onnx",
    "funasr",
    "kaldi-native-fbank",
    "onnx",
    "onnxruntime-gpu",
    "pypinyin",
    "scipy",
    "soundfile",
    "librosa",
    "modelscope",
    "huggingface_hub",
    "faster-whisper",
    "hyperpyyaml",
    "conformer",
    "omegaconf",
    "hydra-core",
    "diffusers",
    "lightning",
    "inflect",
    "wetext",
]
subprocess.run(["pip", "install", "-q", *pip_pkgs], check=True)

print("=== 2. 准备 CosyVoice2 源码与 Matcha-TTS 子模块 ===")
COSY_REPO = Path("/content/CosyVoice")
if not COSY_REPO.is_dir():
    subprocess.run([
        "git", "clone", "--depth", "1", "--recursive",
        "https://github.com/FunAudioLLM/CosyVoice.git", str(COSY_REPO)
    ], check=True)
os.environ["COSYVOICE_REPO"] = str(COSY_REPO)

print("=== 3. 准备 sherpa-onnx Fun-ASR-Nano 训练定义 (nano.py) ===")
SHERPA_REPO = Path("/content/sherpa-onnx")
if not SHERPA_REPO.is_dir():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/k2-fsa/sherpa-onnx.git", str(SHERPA_REPO)
    ], check=True)
os.environ["DVO_SHERPA_NANO_DIR"] = str(SHERPA_REPO / "scripts/sense-voice/rknn")

print("=== 4. 下载 CosyVoice2-0.5B 与 Fun-ASR-Nano-2512 预训练权重 ===")
from huggingface_hub import snapshot_download

COSY_MODEL_DIR = MODELS_ROOT / "CosyVoice2-0.5B"
if not (COSY_MODEL_DIR / "cosyvoice2.yaml").is_file():
    try:
        snapshot_download("FunAudioLLM/CosyVoice2-0.5B", local_dir=str(COSY_MODEL_DIR))
    except Exception:
        from modelscope import snapshot_download as ms_download
        ms_download("iic/CosyVoice2-0.5B", local_dir=str(COSY_MODEL_DIR))

NANO_PRETRAINED_DIR = MODELS_ROOT / "Fun-ASR-Nano-2512"
if not (NANO_PRETRAINED_DIR / "model.pt").is_file():
    snapshot_download("FunAudioLLM/Fun-ASR-Nano-2512", local_dir=str(NANO_PRETRAINED_DIR))

print("=== 5. 下载原始未微调 sherpa-onnx SenseVoice 独立裁判模型 & tokens.txt ===")
JUDGE_MODEL_DIR = MODELS_ROOT / "sherpa-onnx-sense-voice-funasr-nano-int8-2025-12-17"
if not (JUDGE_MODEL_DIR / "model.int8.onnx").is_file():
    tar_url = (
        "https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/"
        "sherpa-onnx-sense-voice-funasr-nano-int8-2025-12-17.tar.bz2"
    )
    tar_path = MODELS_ROOT / "sense-voice-nano.tar.bz2"
    urllib.request.urlretrieve(tar_url, str(tar_path))
    with tarfile.open(tar_path, "r:bz2") as tf:
        tf.extractall(MODELS_ROOT)
    tar_path.unlink(missing_ok=True)

os.environ["DVO_SHERPA_TOKENS"] = str(JUDGE_MODEL_DIR / "tokens.txt")
print("\n✅ 所有依赖与三大基础模型准备完毕！")
print("  - CosyVoice2-0.5B :", COSY_MODEL_DIR)
print("  - Fun-ASR-Nano    :", NANO_PRETRAINED_DIR)
print("  - 独立裁判模型     :", JUDGE_MODEL_DIR)

## Cell 3（先试听把关）：合成 25 条 Pilot 试听小样 + 渲染播放器与质检报告

In [ ]:
import json
import subprocess
from pathlib import Path
from IPython.display import Audio, HTML, display

PILOT_OUT = DRIVE_ROOT / "m3_tts_prep" / "pilot_audio"
PROMPT_POOL_JSONL = DRIVE_ROOT / "m3_tts_prep" / "prompt_pool" / "prompt_pool.jsonl"
PILOT_PROMPTS_JSONL = DRIVE_ROOT / "m3_tts_prep" / "pilot_prompts_25.jsonl"

cmd = [
    "python", str(REPO_DIR / "training/asr/synthesize_personal_tts.py"),
    "--mode", "pilot",
    "--prompts", str(PILOT_PROMPTS_JSONL),
    "--prompt-pool", str(PROMPT_POOL_JSONL),
    "--cosyvoice-model", str(COSY_MODEL_DIR),
    "--cosyvoice-repo", str(COSY_REPO),
    "--judge-model-dir", str(JUDGE_MODEL_DIR),
    "--out-dir", str(PILOT_OUT),
    "--personal-ratio", "1.0",  # Pilot 阶段 100% 试听本人克隆音色
]
subprocess.run(cmd, check=True)

report = json.loads((PILOT_OUT / "quality_report.json").read_text("utf-8"))
print("\n📊 25 条 Pilot 闭环质检汇总:", json.dumps(report, ensure_ascii=False, indent=2))

# 先展示 1 条本人真实参考录音做 AB 对照
ref_first = json.loads(PROMPT_POOL_JSONL.read_text("utf-8").splitlines()[0])
ref_wav = PROMPT_POOL_JSONL.parent / ref_first["wav_path"]
display(HTML(f"<h4>🎧 本人原始参考音频对照（{ref_first['id']}）：<code>{ref_first['prompt_text']}</code></h4>"))
display(Audio(filename=str(ref_wav)))

# 展示通过质检的克隆试听小样
display(HTML("<h4>🔊 CosyVoice2 零样本克隆 + SenseVoice 独立裁判通过样本试听（前 15 条）：</h4>"))
manifest_lines = (PILOT_OUT / "manifest.jsonl").read_text("utf-8").splitlines()
for line in manifest_lines[:15]:
    rec = json.loads(line)
    wav_file = PILOT_OUT / rec["audio"]
    display(HTML(
        f"<div style='margin-top:8px;padding:6px;border-left:3px solid #2b8a3e;background:#f8f9fa;'>"
        f"<b>[{rec['id']}]</b> 目标文本：<code>{rec['prompt']}</code><br/>"
        f"裁判识别：<code>{rec['judge_text']}</code> | CER={rec['judge_cer']} | "
        f"语速={rec['chars_per_sec']} 字/秒 | 时长={rec['duration_s']}s | 参考源={rec['ref_prompt_id']} (第{rec['attempt']}次通过)"
        f"</div>"
    ))
    display(Audio(filename=str(wav_file)))

## Cell 4（可选）：处理今晚戴耳机游戏背景录音（Silero VAD + Whisper/SenseVoice 双模型交叉伪标注）

In [ ]:
import subprocess
from pathlib import Path

# 将今晚录制的会话文件夹上传到云盘 `game_sessions/` 即可自动检测并清洗
GAME_SESSIONS_IN = DRIVE_ROOT / "game_sessions"
GAME_MINED_OUT = DRIVE_ROOT / "m3_tts_prep" / "game_mined"

if GAME_SESSIONS_IN.is_dir() and any(GAME_SESSIONS_IN.iterdir()):
    print(f"🎮 检测到游戏录音目录 {GAME_SESSIONS_IN}，启动 VAD 切分 + 双模型交叉校验伪标注...")
    subprocess.run([
        "python", str(REPO_DIR / "tools/mine_game_sessions.py"),
        "--sessions", str(GAME_SESSIONS_IN),
        "--out", str(GAME_MINED_OUT),
        "--vad-model", str(REPO_DIR / "models/silero_vad.int8.onnx"),
        "--transcribe",
        "--sensevoice-dir", str(JUDGE_MODEL_DIR),
        "--whisper-model", "large-v3",
        "--device", "cuda",
    ], check=True)
else:
    print(f"ℹ️ 未检测到 {GAME_SESSIONS_IN}（若今晚尚未上传游戏会话可直接跳过此步，不影响后续扩词微调）。")

## Cell 5（批量扩词造数）：全量 3,600 条克隆合成 + 独立裁判闭环过滤（支持断点续跑）

In [ ]:
import subprocess

FULL_PROMPTS_JSONL = DRIVE_ROOT / "m3_tts_prep" / "full_prompts_3600.jsonl"
SYNTH_DATASET_OUT = DRIVE_ROOT / "m3_tts_prep" / "synth_dataset_3600"

# --personal-ratio 0.80: 80% 本人音色克隆 + 20% 语速/辅助说话人多样化，防止 ASR 过拟合单一声码器指纹
# --resume: 若 Colab 中途断开，重跑本单元会直接从已完成的条目继续合成
subprocess.run([
    "python", str(REPO_DIR / "training/asr/synthesize_personal_tts.py"),
    "--mode", "full",
    "--resume",
    "--prompts", str(FULL_PROMPTS_JSONL),
    "--prompt-pool", str(PROMPT_POOL_JSONL),
    "--personal-ratio", "0.80",
    "--cosyvoice-model", str(COSY_MODEL_DIR),
    "--cosyvoice-repo", str(COSY_REPO),
    "--judge-model-dir", str(JUDGE_MODEL_DIR),
    "--out-dir", str(SYNTH_DATASET_OUT),
], check=True)

## Cell 6：微调与导出 `asr-personal-v2`（真机主锚点 + 动态音乐混音 + INT8 导出 + 离线对比评测）

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

DATASET_DIR = DRIVE_ROOT / "dataset"
assert (DATASET_DIR / "manifest.jsonl").is_file(), (
    f"未找到 {DATASET_DIR / 'manifest.jsonl'}！请先确认本机 data/dataset/manifest.jsonl 已同步到 Google Drive。"
)

EXP_DIR = Path("/content/exp/asr-personal-v2")
FINAL_EXPORT_DRIVE = DRIVE_ROOT / "models" / "asr-personal-v2"

extra_manifests = []
synth_mf = DRIVE_ROOT / "m3_tts_prep" / "synth_dataset_3600" / "manifest.jsonl"
if synth_mf.is_file():
    extra_manifests.append(str(synth_mf))

game_mf = DRIVE_ROOT / "m3_tts_prep" / "game_mined" / "manifest.jsonl"
if game_mf.is_file():
    extra_manifests.append(str(game_mf))

print("=== 1. 启动 Fun-ASR-Nano CTC 个性化精调 ===")
print("附加语料清单:", extra_manifests)
train_cmd = [
    "python", str(REPO_DIR / "training/asr/finetune.py"),
    "--dataset", str(DATASET_DIR),
    "--pretrained", str(NANO_PRETRAINED_DIR),
    "--sherpa-nano-dir", os.environ["DVO_SHERPA_NANO_DIR"],
    "--tokens", os.environ["DVO_SHERPA_TOKENS"],
    "--aishell-utterances", "0",  # 如云盘已放 AISHELL-1 可改为 3000 并传 --aishell
    "--repeat", "4",
    "--tts-max-per-epoch", "1800",
    "--num-epochs", "4",
    "--lr", "8e-5",
    "--unfreeze-encoder-layers", "3",
    "--encoder-lr", "1.5e-5",
    "--mix-probability", "0.5",
    "--snr", "0", "15",
    "--exp-dir", str(EXP_DIR),
]
if extra_manifests:
    train_cmd += ["--extra-manifests", *extra_manifests]

subprocess.run(train_cmd, check=True)

print("\n=== 2. 逐轮导出 QUInt8 ONNX 并运行离线评测对比 ===")
extra_test_mfs = [str(game_mf)] if game_mf.is_file() else []

# 先评测原始未微调基线模型
orig_eval_json = EXP_DIR / "eval-baseline-orig.json"
eval_base_cmd = [
    "python", str(REPO_DIR / "training/asr/evaluate.py"),
    "--model-dir", str(JUDGE_MODEL_DIR),
    "--dataset", str(DATASET_DIR),
    "--out", str(orig_eval_json),
]
if extra_test_mfs:
    eval_base_cmd += ["--extra-test-manifests", *extra_test_mfs]
print("--- Baseline (Original SenseVoice-Nano) ---")
subprocess.run(eval_base_cmd, check=True)

best_epoch = None
best_score = 1e9

for epoch in range(1, 5):
    ckpt = EXP_DIR / f"epoch-{epoch}.pt"
    if not ckpt.is_file():
        continue
    export_dir = EXP_DIR / f"export-epoch-{epoch}"
    subprocess.run([
        "python", str(REPO_DIR / "training/asr/export.py"),
        "--checkpoint", str(ckpt),
        "--out", str(export_dir),
        "--sherpa-nano-dir", os.environ["DVO_SHERPA_NANO_DIR"],
        "--tokens", os.environ["DVO_SHERPA_TOKENS"],
        "--comment", f"Fun-ASR-Nano-2512 CTC branch, asr-personal-v2 epoch {epoch}",
    ], check=True)

    eval_out = EXP_DIR / f"eval-epoch-{epoch}.json"
    ev_cmd = [
        "python", str(REPO_DIR / "training/asr/evaluate.py"),
        "--model-dir", str(export_dir),
        "--dataset", str(DATASET_DIR),
        "--out", str(eval_out),
    ]
    if extra_test_mfs:
        ev_cmd += ["--extra-test-manifests", *extra_test_mfs]
    print(f"--- Epoch {epoch} ---")
    subprocess.run(ev_cmd, check=True)

    rep = json.loads(eval_out.read_text("utf-8"))
    cer_q = rep["cer"].get("quiet", 0.0)
    cer_m = rep["cer"].get("music", 0.0)
    bg_fp = rep.get("background_clips_with_command_phrase", 0)
    composite = cer_q + cer_m + bg_fp * 0.5
    if composite < best_score:
        best_score = composite
        best_epoch = epoch

assert best_epoch is not None, "未找到导出的训练轮次检查点！"
best_export = EXP_DIR / f"export-epoch-{best_epoch}"
FINAL_EXPORT_DRIVE.mkdir(parents=True, exist_ok=True)
for fname in ["model.int8.onnx", "model.onnx", "tokens.txt"]:
    src_f = best_export / fname
    if src_f.is_file():
        shutil.copy2(src_f, FINAL_EXPORT_DRIVE / fname)
shutil.copy2(EXP_DIR / "summary.json", FINAL_EXPORT_DRIVE / "train_summary.json")
shutil.copy2(EXP_DIR / f"eval-epoch-{best_epoch}.json", FINAL_EXPORT_DRIVE / "eval_report.json")

print(f"\n🎉 已自动选出最优轮次 Epoch {best_epoch} 并同步至 Google Drive:")
print(f"   {FINAL_EXPORT_DRIVE}")
print("   本机只需将 `model.int8.onnx` 与 `tokens.txt` 放入 `models/asr-personal-v2/` 并更新 `config/local.toml` 即可生效！")